In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

import sys

import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.nn.functional import interpolate

sys.path.insert(0, "../../../src")

from juart.conopt.functional.fourier import nonuniform_fourier_transform_forward
from juart.parim.analytic import cyclic_head_coil
from juart.phantoms.mni import BrainPhantom5D
from juart.sampling.spherical import spherical_trajectory_3d
from juart.vis.interactive import InteractiveFigure3D, InteractiveFigure4D
import zarr

def nearest_fibonacci(n: int) -> int:
    """Gibt die Fibonacci-Zahl zurück, die am nächsten an n liegt."""
    if n < 0:
        raise ValueError("Bitte gib eine positive Zahl ein.")
    
    # Startwerte der Fibonacci-Folge
    a, b = 0, 1
    
    # Finde die erste Fibonacci-Zahl, die größer oder gleich n ist
    while b < n:
        a, b = b, a + b
    
    # Entscheide, welche der beiden (a oder b) näher an n liegt
    if abs(a - n) <= abs(b - n):
        return a
    else:
        return b

phantom = BrainPhantom5D(
    B0=3,
    B0_shimming=True,
)

dTE, TE0, nTE = 5, 5, 1
dTI, TI0, nTI = 100, 20, 1

TE = TE0 + dTE * np.arange(nTE)
TI = TI0 + dTI * np.arange(nTI)

TR = 1e6
IE = 1

x_image = phantom.signal(TI, TE, TR, IE)
x_image = x_image / np.abs(x_image).max()
x_image = torch.from_numpy(x_image).to(torch.complex64)

x_image = torch.complex(
    interpolate(x_image[None, None, ..., 0, 0].real, (128, 128, 128)),
    interpolate(x_image[None, None, ..., 0, 0].imag, (128, 128, 128)),
)[0, 0, ..., None, None]

spokes = 128

for R in [2,4,8,16,32,64,128]:
    print(R)
    number_points = nearest_fibonacci(spokes*128/R)
    k_unraveled = spherical_trajectory_3d(128,number_points)
    k = k_unraveled.reshape((3, -1))
    C = cyclic_head_coil((8, 128, 128, 128))
    coil_images = C * x_image[None, ..., 0, 0]
    
    d = nonuniform_fourier_transform_forward(
        k,
        coil_images,
        isign=-1,
    )
    
    store = zarr.storage.LocalStore(f"/home/jovyan/datasets/fibo_phantom_{spokes}spk_R{R}_{number_points}points")
    group = zarr.create_group(
        store,
        overwrite=True,
    )
    
    group.create_array(
        "C",
        shape=C.shape,
        dtype=np.complex64,
        overwrite=True,
    )
    group.create_array(
        "k", 
        shape=k.shape,
        dtype=np.float32,
        overwrite=True,
    )
    group.create_array(
        "d", 
        shape=d.shape,
        dtype=np.complex64,
        overwrite=True,
    )
    
    group["C"] = C.numpy()
    group["k"] = k.numpy()
    group["d"] = d.numpy()